# ¿Cuánto vale esta casa?

**Grupo:** Calveyra, Desivo, Giordanelli, Rickert

## Milestone 1 — La tabla de decisiones

Primero cargamos el dataset y lo inspeccionamos para tener la evidencia (tipos, cardinalidad, faltantes) que respalda las decisiones de la tabla.

In [1]:
import pandas as pd

df = pd.read_csv('house-prices-extended.csv')
print(df.shape)
df.head()

(1460, 20)


,ListingId,LotArea,OverallQual,OverallCond,TotalBsmtSF,FullBath,HalfBath,BedroomAbvGr,TotRmsAbvGrd,Fireplaces,GarageArea,YearBuilt,LotFrontage,MonthSold,State,District,HouseStyle,CentralAir,SaleCondition,SalePrice
0,677742,8450,7,5,856,2,1,3,8,0,548,1970,64.0,7,IL,Larkspur,Townhouse,N,Normal,172700
1,551423,9600,6,8,1262,2,0,3,6,1,460,1935,73.0,11,CA,Greenfield,2Story,Y,Normal,369700
2,237268,11250,7,5,920,2,1,3,6,1,608,1949,80.0,9,AZ,Fernwood,1Story,Y,Normal,165700
3,579284,9550,7,5,756,1,0,3,7,1,642,1924,72.0,7,FL,Stonegate,2Story,Y,Normal,164300
4,598125,14260,8,5,1145,2,1,4,9,1,836,1951,87.0,2,NC,Xavier Park,Loft,Y,Normal,283700


In [2]:
resumen = pd.DataFrame({
    'dtype': df.dtypes,
    'n_unicos': df.nunique(),
    'n_faltantes': df.isna().sum(),
    'pct_faltantes': (100 * df.isna().mean()).round(2),
})
resumen

,dtype,n_unicos,n_faltantes,pct_faltantes
ListingId,int64,1460,0,0.00
LotArea,int64,1073,0,0.00
OverallQual,int64,10,0,0.00
OverallCond,int64,9,0,0.00
TotalBsmtSF,int64,721,0,0.00
FullBath,int64,4,0,0.00
HalfBath,int64,3,0,0.00
BedroomAbvGr,int64,8,0,0.00
TotRmsAbvGrd,int64,12,0,0.00
Fireplaces,int64,4,0,0.00


### Tabla de decisiones

| Columna | Qué es | Qué significa la resta entre dos valores | Cardinalidad | ¿Puede faltar? | ¿Disponible al predecir? | Codificación elegida | Motivo |
|---|---|---|---|---|---|---|---|
| `ListingId` | Identificador correlativo de la fila | Nada: es un número de carga, no una magnitud | 1460 (todos únicos) | No | Sí, pero no debe usarse | **Se descarta**, no entra como feature | Es un índice, no información de la casa. Si entra, la red puede "memorizar" el orden de carga en vez de aprender patrones — **trampa 1** (ver Pregunta 3) |
| `LotArea` | Superficie del lote (pies²) | Diferencia real de superficie | Continua, muchos valores | No | Sí | Numérica, escalada | Magnitud continua, la escala favorece el entrenamiento |
| `OverallQual` | Calidad general de material/acabado (1–10) | Un punto más de calidad | 10 | No | Sí | Numérica, escalada | Ordinal con paso uniforme, se puede tratar como continua |
| `OverallCond` | Condición general (1–9) | Un punto más de condición | 9 | No | Sí | Numérica, escalada | Ídem `OverallQual` |
| `TotalBsmtSF` | Superficie de sótano (pies²) | Diferencia real de superficie | Continua | No | Sí | Numérica, escalada | Magnitud continua |
| `FullBath` | Baños completos sobre nivel de suelo | Un baño completo más | 4 | No | Sí | Numérica, escalada | Conteo, la resta es significativa |
| `HalfBath` | Medios baños sobre nivel de suelo | Un medio baño más | 3 | No | Sí | Numérica, escalada | Conteo |
| `BedroomAbvGr` | Dormitorios sobre nivel de suelo | Un dormitorio más | 8 | No | Sí | Numérica, escalada | Conteo |
| `TotRmsAbvGrd` | Habitaciones totales sobre nivel de suelo | Una habitación más | 12 | No | Sí | Numérica, escalada | Conteo |
| `Fireplaces` | Chimeneas | Una chimenea más | 4 | No | Sí | Numérica, escalada | Conteo |
| `GarageArea` | Superficie de garage (pies²) | Diferencia real de superficie | Continua | No | Sí | Numérica, escalada | Magnitud continua |
| `YearBuilt` | Año de construcción | Antigüedad relativa entre dos casas | Continua (años) | No | Sí | Numérica, escalada | El año ya es lineal en el tiempo, no hace falta transformarlo |
| `LotFrontage` | Frente del lote (pies) | Diferencia real de frente | Continua | **Sí, ~7%** | Sí (cuando está) | Numérica escalada + imputación por mediana del conjunto de entrenamiento + indicador binario de faltante | Falta de forma no trivial; comparamos contra imputar con 0 y probamos si el faltante en sí informa (Milestone 2, Pregunta 4) |
| `MonthSold` | Mes de venta (1–12) | Nada lineal: diciembre y enero están a un mes de distancia, no a once | 12 | No | Sí | Seno/coseno (codificación cíclica) | Es una variable **cíclica** — **trampa 2**: tratarla como entero plano le asigna una distancia falsa a los meses que están en los extremos del año |
| `State` | Estado donde está la casa | Nada, es nominal | 12 | No | Sí | One-hot | Cardinalidad baja/moderada, sin orden natural — el one-hot no le impone ninguna relación entre estados |
| `District` | Distrito dentro del estado | Nada, es nominal | 72 | No | Sí | Embedding aprendido | Cardinalidad alta — un one-hot de 72 columnas es viable pero disperso; el embedding aprende una representación densa. Si se codifica como entero 0–71 la red asume un orden y una distancia entre distritos que no existen — **trampa 3** (ver Pregunta 2) |
| `HouseStyle` | Estilo arquitectónico de la casa | Nada, es nominal | 6 | No | Sí | One-hot | Cardinalidad baja, sin orden |
| `CentralAir` | Tiene aire acondicionado central (Y/N) | Nada, es binaria | 2 | No | Sí | Binaria 0/1 | Dos categorías, mapeo directo sin necesidad de one-hot |
| `SaleCondition` | Condición de la venta (normal, embargo, etc.) | Nada, es nominal | 5 | No | Sí | One-hot | Cardinalidad baja, sin orden natural entre condiciones |
| `SalePrice` | Precio de venta (dólares) | Diferencia real de precio | Continua | No | — (es el target) | Salida de la red, no se codifica como entrada | Es lo que queremos predecir: define la capa de salida (activación lineal) y la loss de regresión |





> **Las tres trampas** de este dataset son `ListingId` (identificador tratado como feature), `MonthSold` (variable cíclica tratada como lineal) y `District` (alta cardinalidad tratada como si tuviera orden). Las tres entrenan igual si se las pasa por alto — el problema aparece recién al mirar el error o al llevar el modelo a producción.